In [1]:
%matplotlib inline

# tenth_attempt.ipynb -- Diabetic Retinopathy Detection (CUSTOM category only)

**Changes vs ninth_attempt.ipynb (M-series — custom only):**
- **(M-1) 4-architecture ensemble**: Replace 3-seed same-architecture ensemble with 4 complementary architectures (single seed=42 each). Architectural diversity provides a different and stronger source of ensemble diversity than seed variation on the same model.
- **(M-2) SECustomNetV3 (new)**: Combines SE channel attention (from SECustomNetV2) with residual skip connections (from CustomNetV3). Completes the 2×2 design matrix: attention × skip.
- **(M-3) Uniform mean ensemble**: No val-based weight search. 500 val samples cannot reliably optimise 3 free weights on the simplex — same reasoning as ninth_attempt's rejection of grid search.
- **(M-4) Custom-only notebook**: Fine-tuning handled separately by cris.

**CUSTOM validity**: output_custom.csv = uniform mean of 4 scratch models — no pretrained weights in any architecture.

**Architecture design matrix:**

| Model | Channel attention | Residual skip |
|---|---|---|
| CustomNetV2 | No | No |
| SECustomNetV2 | Yes | No |
| CustomNetV3 | No | Yes |
| **SECustomNetV3** | **Yes** | **Yes** |

## 1. Imports & Setup

In [2]:
from __future__ import print_function, division
import os, csv
import torch
import pandas as pd
from skimage import io, transform, util, color
from sklearn import metrics
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, utils
import torchvision.transforms.functional as TF
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim import lr_scheduler
import time
import copy
from PIL import Image
from zipfile import ZipFile
import random
import numpy.random as npr
import cv2
import warnings

warnings.filterwarnings('ignore')
random.seed(42)
npr.seed(42)
torch.manual_seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

plt.ion()
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

cuda:0


In [3]:
DATA_ROOT = '/kaggle/input/datasets/mariamuozperez/lab5-cv/LS5_CV_2025_2026_DB_Retinopathy'

In [4]:
# Run once to extract data, then comment out
# import zipfile
# with zipfile.ZipFile('./db.zip', 'r') as z:
#     z.extractall('./data')

## 2. Dataset

In [5]:
class RetinopathyDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None, maxSize=0):
        self.dataset = pd.read_csv(csv_file, header=0,
                                   dtype={'id': str, 'eye': int, 'label': int})
        if maxSize > 0:
            idx = np.random.RandomState(seed=42).permutation(range(len(self.dataset)))
            self.dataset = self.dataset.iloc[idx[:maxSize]].reset_index(drop=True)
        self.root_dir = root_dir
        self.img_dir  = os.path.join(root_dir, 'images')
        self.transform = transform
        self.levels  = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative DR']
        self.classes = ['No DR', 'DR']

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()
        img_name = os.path.join(self.img_dir, self.dataset.id[idx] + '.jpg')
        image = io.imread(img_name)
        if self.dataset.eye[idx] == 1:
            image = image[:, ::-1, :]
        sample = {
            'image': image,
            'eye':   self.dataset.eye[idx],
            'label': (self.dataset.label[idx] > 0).astype(dtype=np.int64)
        }
        if self.transform:
            sample = self.transform(sample)
        return sample

## 3. Transforms

In [6]:
class CropByEye(object):
    def __init__(self, threshold, border):
        self.threshold = threshold
        self.border = (border, border) if isinstance(border, int) else border

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        h, w = image.shape[:2]
        imgray = color.rgb2gray(image)
        _, mask = cv2.threshold(imgray, self.threshold, 1, cv2.THRESH_BINARY)
        sidx = np.nonzero(mask)
        if len(sidx[0]) < 20:
            return {'image': image, 'eye': eye, 'label': label}
        minx = np.maximum(sidx[1].min() - self.border[1], 0)
        maxx = np.minimum(sidx[1].max() + 1 + self.border[1], w)
        miny = np.maximum(sidx[0].min() - self.border[0], 0)
        maxy = np.minimum(sidx[0].max() + 1 + self.border[1], h)
        image = image[miny:maxy, minx:maxx, ...]
        return {'image': image, 'eye': eye, 'label': label}


class BenGraham(object):
    def __init__(self, sigmaX=10):
        self.sigmaX = sigmaX

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        if image.dtype == np.uint8:
            img_u8 = image
        else:
            img_u8 = (np.clip(image, 0, 1) * 255).astype(np.uint8)
        blurred  = cv2.GaussianBlur(img_u8, (0, 0), self.sigmaX)
        enhanced = cv2.addWeighted(img_u8, 4, blurred, -4, 128)
        enhanced = np.clip(enhanced, 0, 255).astype(np.uint8)
        mask = np.zeros(enhanced.shape, dtype=np.uint8)
        h, w = enhanced.shape[:2]
        cv2.circle(mask, (w // 2, h // 2), int(0.9 * min(h, w) / 2), (1, 1, 1), -1, 8, 0)
        enhanced = enhanced * mask + 128 * (1 - mask)
        return {'image': enhanced.astype(np.float32) / 255.0, 'eye': eye, 'label': label}


class Rescale(object):
    def __init__(self, output_size):
        self.output_size = output_size

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        h, w = image.shape[:2]
        if isinstance(self.output_size, int):
            new_h = self.output_size * h / w if h > w else self.output_size
            new_w = self.output_size if h > w else self.output_size * w / h
        else:
            new_h, new_w = self.output_size
        image = transform.resize(image, (int(new_h), int(new_w)))
        return {'image': image, 'eye': eye, 'label': label}


class RandomCrop(object):
    def __init__(self, output_size):
        self.output_size = (output_size, output_size) if isinstance(output_size, int) else output_size

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        h, w = image.shape[:2]
        new_h, new_w = self.output_size
        top  = np.random.randint(0, h - new_h) if h > new_h else 0
        left = np.random.randint(0, w - new_w) if w > new_w else 0
        image = image[top:top + new_h, left:left + new_w]
        return {'image': image, 'eye': eye, 'label': label}


class CenterCrop(object):
    def __init__(self, output_size):
        self.output_size = (output_size, output_size) if isinstance(output_size, int) else output_size

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        h, w = image.shape[:2]
        new_h, new_w = self.output_size
        top  = int((h - new_h) / 2) if h > new_h else 0
        left = int((w - new_w) / 2) if w > new_w else 0
        image = image[top:top + new_h, left:left + new_w]
        return {'image': image, 'eye': eye, 'label': label}


class ToTensor(object):
    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        image = torch.from_numpy(image.transpose((2, 0, 1)))
        label = torch.tensor(label, dtype=torch.long)
        return {'image': image, 'eye': eye, 'label': label}


class Normalize(object):
    def __init__(self, mean, std):
        self.mean = np.array(mean)
        self.std  = np.array(std)

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        dtype = image.dtype
        mean = torch.as_tensor(self.mean, dtype=dtype, device=image.device)
        std  = torch.as_tensor(self.std,  dtype=dtype, device=image.device)
        image.sub_(mean[:, None, None]).div_(std[:, None, None])
        return {'image': image, 'eye': eye, 'label': label}


class TVRandomHorizontalFlip(object):
    def __init__(self, p=0.5):
        self.flip = transforms.RandomHorizontalFlip(p=p)

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        pil = Image.fromarray(util.img_as_ubyte(image))
        image = util.img_as_float(np.asarray(self.flip(pil)))
        return {'image': image, 'eye': eye, 'label': label}


class TVRandomVerticalFlip(object):
    # (L-3) Align training distribution with the vflip pass in 4-pass TTA.
    def __init__(self, p=0.5):
        self.flip = transforms.RandomVerticalFlip(p=p)

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        pil = Image.fromarray(util.img_as_ubyte(image))
        image = util.img_as_float(np.asarray(self.flip(pil)))
        return {'image': image, 'eye': eye, 'label': label}


class TVRandomRotation(object):
    def __init__(self, degrees=15):
        self.rotate = transforms.RandomRotation(degrees=degrees)

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        pil = Image.fromarray(util.img_as_ubyte(image))
        image = util.img_as_float(np.asarray(self.rotate(pil)))
        return {'image': image, 'eye': eye, 'label': label}


class TVColorJitter(object):
    def __init__(self, brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05):
        self.jitter = transforms.ColorJitter(
            brightness=brightness, contrast=contrast,
            saturation=saturation, hue=hue)

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        pil = Image.fromarray(util.img_as_ubyte(image))
        image = util.img_as_float(np.asarray(self.jitter(pil)))
        return {'image': image, 'eye': eye, 'label': label}

## 4. Data Pipelines & DataLoaders

In [7]:
pixel_mean = [0.485, 0.456, 0.406]
pixel_std  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    CropByEye(0.10, 1),
    BenGraham(sigmaX=10),
    Rescale(256),
    TVRandomHorizontalFlip(p=0.5),
    TVRandomVerticalFlip(p=0.5),
    TVRandomRotation(degrees=15),
    TVColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    RandomCrop(224),
    ToTensor(),
    Normalize(mean=pixel_mean, std=pixel_std),
])

eval_transform = transforms.Compose([
    CropByEye(0.10, 1),
    BenGraham(sigmaX=10),
    Rescale(256),
    CenterCrop(224),
    ToTensor(),
    Normalize(mean=pixel_mean, std=pixel_std),
])

train_dataset = RetinopathyDataset(
    csv_file=os.path.join(DATA_ROOT, 'train.csv'),
    root_dir=DATA_ROOT, maxSize=0, transform=train_transform)
val_dataset = RetinopathyDataset(
    csv_file=os.path.join(DATA_ROOT, 'val.csv'),
    root_dir=DATA_ROOT, transform=eval_transform)
test_dataset = RetinopathyDataset(
    csv_file=os.path.join(DATA_ROOT, 'test.csv'),
    root_dir=DATA_ROOT, transform=eval_transform)
print(f'Train: {len(train_dataset)}  Val: {len(val_dataset)}  Test: {len(test_dataset)}')

Train: 2000  Val: 500  Test: 1000


In [8]:
_sample = train_dataset[0]
_img = _sample['image'].numpy().transpose(1, 2, 0)
print(f'Pipeline output -- dtype: {_img.dtype}, min: {_img.min():.3f}, max: {_img.max():.3f}, mean: {_img.mean():.3f}')
assert abs(_img.mean()) < 0.3, f'BenGraham all-gray bug! mean={_img.mean():.3f}'
print('Sanity check passed.')

Pipeline output -- dtype: float64, min: -2.118, max: 1.763, mean: -0.106
Sanity check passed.


In [9]:
train_labels_bin_for_sampler = (train_dataset.dataset['label'].values > 0).astype(int)
class_counts   = np.bincount(train_labels_bin_for_sampler)
sample_weights = np.where(train_labels_bin_for_sampler == 1,
                          1.0 / class_counts[1], 1.0 / class_counts[0])
sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.float),
    num_samples=len(train_dataset), replacement=True)

train_dataloader = DataLoader(train_dataset, batch_size=64,  sampler=sampler,  num_workers=0)
val_dataloader   = DataLoader(val_dataset,   batch_size=256, shuffle=False, num_workers=0)
test_dataloader  = DataLoader(test_dataset,  batch_size=256, shuffle=False, num_workers=0)

train_labels_bin = (train_dataset.dataset['label'].values > 0).astype(int)
n_neg = int((train_labels_bin == 0).sum())
n_pos = int((train_labels_bin == 1).sum())
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float).to(device)
print(f'No-DR: {n_neg}  DR: {n_pos}  pos_weight: {pos_weight.item():.3f}')
assert abs(pos_weight.item() - 2.759) < 0.05, f'Unexpected pos_weight: {pos_weight.item():.3f}'

criterion      = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
image_datasets = {'train': train_dataset, 'val': val_dataset}
dataloaders    = {'train': train_dataloader, 'val': val_dataloader}
dataset_sizes  = {'train': len(train_dataset), 'val': len(val_dataset)}
class_names    = train_dataset.classes

No-DR: 1468  DR: 532  pos_weight: 2.759


## 5. Training & Evaluation Utilities

In [10]:
def train_model(model, criterion, optimizer, scheduler, num_epochs=25, patience=7, label_smoothing=0.0):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_auc, best_epoch, no_improve = 0.0, -1, 0

    for epoch in range(num_epochs):
        print('Epoch {}/{}'.format(epoch, num_epochs - 1))
        print('-' * 10)
        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()
            numSamples = dataset_sizes[phase]
            outputs_m  = np.zeros((numSamples,), dtype=float)
            labels_m   = np.zeros((numSamples,), dtype=int)
            running_loss, contSamples = 0.0, 0
            for sample in dataloaders[phase]:
                inputs    = sample['image'].to(device).float()
                labels    = sample['label'].to(device).float()
                batchSize = labels.shape[0]
                optimizer.zero_grad()
                with torch.set_grad_enabled(phase == 'train'):
                    logits = model(inputs).flatten()
                    if label_smoothing > 0.0 and phase == 'train':
                        labels_ls = labels * (1 - label_smoothing) + label_smoothing / 2.0
                        loss = criterion(logits, labels_ls)
                    else:
                        loss = criterion(logits, labels)
                    scores = torch.sigmoid(logits).detach()
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                running_loss += loss.item() * batchSize
                outputs_m[contSamples:contSamples + batchSize] = scores.cpu().numpy()
                labels_m [contSamples:contSamples + batchSize] = labels.cpu().numpy()
                contSamples += batchSize
            if phase == 'train':
                scheduler.step()
            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_auc  = metrics.roc_auc_score(labels_m, outputs_m)
            print('{} Loss: {:.4f}  AUC: {:.4f}'.format(phase, epoch_loss, epoch_auc))
            if phase == 'val':
                if epoch_auc > best_auc:
                    best_auc, best_epoch, no_improve = epoch_auc, epoch, 0
                    best_model_wts = copy.deepcopy(model.state_dict())
                else:
                    no_improve += 1
                    if no_improve >= patience:
                        print(f'Early stopping: no improvement for {patience} epochs.')
                        model.load_state_dict(best_model_wts)
                        return model
        print()
    elapsed = time.time() - since
    print('Training complete in {:.0f}m {:.0f}s'.format(elapsed // 60, elapsed % 60))
    print('Best model: epoch {:d}  val AUC: {:.4f}'.format(best_epoch, best_auc))
    model.load_state_dict(best_model_wts)
    return model

In [11]:
def eval_val_auc(model, name, tta=False):
    model.eval()
    n = len(val_dataset)
    scores_m = np.zeros((n, 1), dtype=float)
    labels_m = np.zeros((n,), dtype=int)
    cont = 0
    with torch.no_grad():
        for sample in val_dataloader:
            inputs = sample['image'].to(device).float()
            bs = inputs.shape[0]
            if tta:
                s1 = torch.sigmoid(model(inputs))
                s2 = torch.sigmoid(model(torch.flip(inputs, dims=[3])))    # hflip
                s3 = torch.sigmoid(model(torch.flip(inputs, dims=[2])))    # vflip
                s4 = torch.sigmoid(model(torch.flip(inputs, dims=[2, 3]))) # rot180
                out = (s1 + s2 + s3 + s4) / 4.0
            else:
                out = torch.sigmoid(model(inputs))
            scores_m[cont:cont + bs, :] = out.cpu().numpy()
            labels_m[cont:cont + bs]     = sample['label'].numpy()
            cont += bs
    auc    = metrics.roc_auc_score(labels_m, scores_m)
    suffix = ' (TTA-4)' if tta else ''
    print(f'{name}{suffix}  --  val AUC: {auc:.4f}')
    return auc


def get_val_scores(model, tta=False):
    model.eval()
    n = len(val_dataset)
    scores_m = np.zeros((n, 1), dtype=float)
    labels_m = np.zeros((n,), dtype=int)
    cont = 0
    with torch.no_grad():
        for sample in val_dataloader:
            inputs = sample['image'].to(device).float()
            bs = inputs.shape[0]
            if tta:
                s1 = torch.sigmoid(model(inputs))
                s2 = torch.sigmoid(model(torch.flip(inputs, dims=[3])))
                s3 = torch.sigmoid(model(torch.flip(inputs, dims=[2])))
                s4 = torch.sigmoid(model(torch.flip(inputs, dims=[2, 3])))
                out = (s1 + s2 + s3 + s4) / 4.0
            else:
                out = torch.sigmoid(model(inputs))
            scores_m[cont:cont + bs, :] = out.cpu().numpy()
            labels_m[cont:cont + bs]     = sample['label'].numpy()
            cont += bs
    return scores_m, labels_m


def test_model(model, tta=False):
    model.eval()
    n = len(test_dataset)
    outputs_m = np.zeros((n, 1), dtype=float)
    cont = 0
    with torch.no_grad():
        for sample in test_dataloader:
            inputs = sample['image'].to(device).float()
            bs = inputs.shape[0]
            if tta:
                s1 = torch.sigmoid(model(inputs))
                s2 = torch.sigmoid(model(torch.flip(inputs, dims=[3])))
                s3 = torch.sigmoid(model(torch.flip(inputs, dims=[2])))
                s4 = torch.sigmoid(model(torch.flip(inputs, dims=[2, 3])))
                out = (s1 + s2 + s3 + s4) / 4.0
            else:
                out = torch.sigmoid(model(inputs))
            outputs_m[cont:cont + bs, :] = out.cpu().numpy()
            cont += bs
    return outputs_m

---
## 6. Architecture Definitions

All four models share the same channel progression (3→32→64→128→256→256), GAP, and classifier head
(Dropout(0.2)→Linear(256,128)→ReLU→Dropout(0.4)→Linear(128,1)).
No pretrained weights in any architecture — all are valid for the CUSTOM category.

In [12]:
class SEBlock(nn.Module):
    def __init__(self, channels, r=8):
        super().__init__()
        mid = max(channels // r, 4)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc  = nn.Sequential(
            nn.Linear(channels, mid),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        w = self.gap(x).flatten(1)
        w = self.fc(w).view(x.size(0), x.size(1), 1, 1)
        return x * w

### Architecture A: CustomNetV2 — plain CNN (no attention, no skip)

In [13]:
class CustomNetV2(nn.Module):
    def __init__(self):
        super().__init__()

        def _block(cin, cout):
            return nn.Sequential(
                nn.Conv2d(cin, cout, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(cout),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2, 2),
            )

        self.features = nn.Sequential(
            _block(3,   32),
            _block(32,  64),
            _block(64,  128),
            _block(128, 256),
            _block(256, 256),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.2),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.4),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x).flatten(1)
        return self.classifier(x)

### Architecture B: SECustomNetV2 — channel attention, no skip

In [14]:
class SECustomNetV2(nn.Module):
    def __init__(self):
        super().__init__()

        def _block(cin, cout):
            return nn.Sequential(
                nn.Conv2d(cin, cout, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(cout),
                nn.ReLU(inplace=True),
                SEBlock(cout),
                nn.MaxPool2d(2, 2),
            )

        self.features = nn.Sequential(
            _block(3,   32),
            _block(32,  64),
            _block(64,  128),
            _block(128, 256),
            _block(256, 256),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.2),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.4),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x).flatten(1)
        return self.classifier(x)

### Architecture C: CustomNetV3 — residual skip, no attention

In [15]:
class ResBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(cin, cout, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(cout),
        )
        self.skip = nn.Sequential(
            nn.Conv2d(cin, cout, kernel_size=1, bias=False),
            nn.BatchNorm2d(cout),
        ) if cin != cout else nn.Identity()
        self.relu = nn.ReLU(inplace=True)
        self.pool = nn.MaxPool2d(2, 2)

    def forward(self, x):
        out = self.main(x) + self.skip(x)
        return self.pool(self.relu(out))


class CustomNetV3(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            ResBlock(3,   32),
            ResBlock(32,  64),
            ResBlock(64,  128),
            ResBlock(128, 256),
            ResBlock(256, 256),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.2),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.4),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x).flatten(1)
        return self.classifier(x)

### Architecture D: SECustomNetV3 — residual skip + channel attention (new)

Combines both inductive biases: SE recalibrates channels after the residual fusion+activation,
then MaxPool downsamples. Skip always uses 1×1 projection when cin != cout (Identity for 256→256).

In [16]:
class SEResBlock(nn.Module):
    def __init__(self, cin, cout, r=8):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(cin, cout, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(cout),
        )
        self.skip = nn.Sequential(
            nn.Conv2d(cin, cout, kernel_size=1, bias=False),
            nn.BatchNorm2d(cout),
        ) if cin != cout else nn.Identity()
        self.relu = nn.ReLU(inplace=True)
        self.se   = SEBlock(cout, r=r)
        self.pool = nn.MaxPool2d(2, 2)

    def forward(self, x):
        out = self.relu(self.main(x) + self.skip(x))
        out = self.se(out)   # channel attention after residual fusion + activation
        return self.pool(out)


class SECustomNetV3(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            SEResBlock(3,   32),
            SEResBlock(32,  64),
            SEResBlock(64,  128),
            SEResBlock(128, 256),
            SEResBlock(256, 256),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.2),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.4),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x).flatten(1)
        return self.classifier(x)

---
## 7. Smoke Tests

Verify all four architectures produce the correct output shape before committing to training.

In [17]:
_x = torch.randn(2, 3, 224, 224).to(device)

for name, cls in [('CustomNetV2', CustomNetV2),
                   ('SECustomNetV2', SECustomNetV2),
                   ('CustomNetV3', CustomNetV3),
                   ('SECustomNetV3', SECustomNetV3)]:
    net = cls().to(device)
    with torch.no_grad():
        out = net(_x)
    assert out.shape == (2, 1), f'{name}: expected (2,1), got {out.shape}'
    total = sum(p.numel() for p in net.parameters())
    assert total < 2_000_000, f'{name}: param count {total:,} exceeds 2M'
    print(f'{name}  output: {out.shape}  params: {total:,}')
    del net

del _x
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('All smoke tests passed.')

CustomNetV2  output: torch.Size([2, 1])  params: 1,012,257
SECustomNetV2  output: torch.Size([2, 1])  params: 1,051,229
CustomNetV3  output: torch.Size([2, 1])  params: 1,056,321
SECustomNetV3  output: torch.Size([2, 1])  params: 1,095,293
All smoke tests passed.


---
## 8. Training Loop — 4 architectures, seed=42

Each architecture is trained from scratch with the same seed (42) and identical hyperparameters.
Diversity comes from architecture differences, not random variation.
Models are deleted after inference to free GPU memory.

In [18]:
SEED = 42

def set_seed(seed):
    random.seed(seed)
    npr.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

arch_configs = {
    'v2':    CustomNetV2,
    'se_v2': SECustomNetV2,
    'v3':    CustomNetV3,
    'se_v3': SECustomNetV3,
}
arch_names = ['v2', 'se_v2', 'v3', 'se_v3']

In [19]:
val_aucs       = {}
val_scores     = {}
test_scores    = {}
val_labels_ref = None

for arch_name in arch_names:
    sep = '=' * 60
    print('\n' + sep)
    print(f'Training {arch_name}  seed={SEED}')
    print(sep)
    set_seed(SEED)

    model     = arch_configs[arch_name]().to(device)
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=5e-3)
    scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)
    model = train_model(model, criterion, optimizer, scheduler,
                        num_epochs=50, patience=10, label_smoothing=0.0)

    torch.save(model.state_dict(), f'best_{arch_name}.pth')
    print(f'Saved: best_{arch_name}.pth')

    auc = eval_val_auc(model, arch_name, tta=True)
    val_aucs[arch_name] = auc

    vs, vl = get_val_scores(model, tta=True)
    val_scores[arch_name] = vs
    if val_labels_ref is None:
        val_labels_ref = vl

    test_scores[arch_name] = test_model(model, tta=True)

    del model, optimizer, scheduler
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


Training v2  seed=42
Epoch 0/49
----------
train Loss: 1.1197  AUC: 0.5016
val Loss: 1.2310  AUC: 0.4521

Epoch 1/49
----------
train Loss: 1.1070  AUC: 0.5019
val Loss: 1.2504  AUC: 0.4535

Epoch 2/49
----------
train Loss: 1.1026  AUC: 0.5049
val Loss: 1.1890  AUC: 0.4681

Epoch 3/49
----------
train Loss: 1.1011  AUC: 0.5022
val Loss: 1.1976  AUC: 0.4758

Epoch 4/49
----------
train Loss: 1.0947  AUC: 0.4933
val Loss: 1.2222  AUC: 0.5119

Epoch 5/49
----------
train Loss: 1.1010  AUC: 0.5157
val Loss: 1.1492  AUC: 0.5742

Epoch 6/49
----------
train Loss: 1.0809  AUC: 0.5523
val Loss: 1.1802  AUC: 0.5977

Epoch 7/49
----------
train Loss: 1.0627  AUC: 0.5988
val Loss: 1.1503  AUC: 0.6494

Epoch 8/49
----------
train Loss: 1.0448  AUC: 0.6338
val Loss: 1.0134  AUC: 0.6796

Epoch 9/49
----------
train Loss: 1.0276  AUC: 0.6501
val Loss: 1.3327  AUC: 0.6880

Epoch 10/49
----------
train Loss: 1.0434  AUC: 0.6288
val Loss: 1.0220  AUC: 0.6818

Epoch 11/49
----------
train Loss: 1.0072 

---
## 9. Ensemble & Validation Summary

Uniform mean across all 4 architectures — no val-based weight optimisation.
Val AUC is reported for diagnostic purposes only; no decisions are made from it at this stage.

In [20]:
print('=== Per-architecture validation AUC (TTA-4) ===')
for arch_name in arch_names:
    print(f'  {arch_name}: {val_aucs[arch_name]:.4f}')

mean_val_scores  = np.mean(np.stack([val_scores[k] for k in arch_names], axis=0), axis=0)
auc_ensemble_val = metrics.roc_auc_score(val_labels_ref, mean_val_scores)
print(f'\n  4-arch uniform ensemble val AUC (TTA-4): {auc_ensemble_val:.4f}')

outputs_custom = np.mean(np.stack([test_scores[k] for k in arch_names], axis=0), axis=0)

assert outputs_custom.shape == (1000, 1)
assert np.isfinite(outputs_custom).all()
print(f'\nCustom score range: [{outputs_custom.min():.4f}, {outputs_custom.max():.4f}]')
print('Checks passed.')

=== Per-architecture validation AUC (TTA-4) ===
  v2: 0.7788
  se_v2: 0.5241
  v3: 0.7688
  se_v3: 0.7711

  4-arch uniform ensemble val AUC (TTA-4): 0.7837

Custom score range: [0.5033, 0.9305]
Checks passed.


---
## 10. Generate output_custom.csv

**CUSTOM category**: uniform mean of CustomNetV2, SECustomNetV2, CustomNetV3, SECustomNetV3 (all scratch, seed=42, TTA-4).

In [21]:
with open('output_custom.csv', mode='w', newline='') as f:
    csv.writer(f).writerows(outputs_custom)
print('Written: output_custom.csv')

Written: output_custom.csv


In [22]:
with ZipFile('./codabench_submission.zip', 'w') as zf:
    zf.write('./output_custom.csv')
print('Created: codabench_submission.zip')

print('\nFinal summary (tenth_attempt.ipynb):')
for arch_name in arch_names:
    print(f'  {arch_name}  val AUC (TTA-4): {val_aucs[arch_name]:.4f}')
print(f'  4-arch ensemble val AUC (TTA-4): {auc_ensemble_val:.4f}')

Created: codabench_submission.zip

Final summary (tenth_attempt.ipynb):
  v2  val AUC (TTA-4): 0.7788
  se_v2  val AUC (TTA-4): 0.5241
  v3  val AUC (TTA-4): 0.7688
  se_v3  val AUC (TTA-4): 0.7711
  4-arch ensemble val AUC (TTA-4): 0.7837
